# 📚 Technique 56: Semantic Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/56_semantic_search.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 56
**Difficulty:** Intermediate

## 📋 Description

**Semantic Search** uses vector embeddings to find documents based on meaning rather than keyword matching. By converting text into high-dimensional vectors that capture semantic relationships, it enables finding relevant content even when query and document use different words to express the same concept.

### When to Use:
- When users may use **different terminology** than the documents
- For **conceptual queries** without exact keyword matches
- When you need to handle **synonyms and paraphrases**
- For **multilingual search** across languages
- When keyword search produces too many **false negatives**

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                   SEMANTIC SEARCH PIPELINE                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  INDEXING PHASE                                                 │
│  ┌─────────────┐    ┌──────────────┐    ┌─────────────────┐    │
│  │  Document   │───▶│  Embedding   │───▶│  Vector Store   │    │
│  │    Text     │    │    Model     │    │   (Index)       │    │
│  └─────────────┘    └──────────────┘    └─────────────────┘    │
│         │                  │                     │              │
│         │                  ▼                     ▼              │
│         │         ┌─────────────────┐    ┌──────────────┐       │
│         │         │  1536-dim       │    │  FAISS/Pine- │       │
│         │         │  Vector         │    │  cone/Weavi- │       │
│         │         │  Representation │    │  ate/etc     │       │
│         │         └─────────────────┘    └──────────────┘       │
│                                                                 │
│  QUERY PHASE                                                    │
│  ┌─────────────┐    ┌──────────────┐    ┌─────────────────┐    │
│  │   User      │───▶│  Embedding   │───▶│  Similarity     │    │
│  │   Query     │    │    Model     │    │  Search (k-NN)  │    │
│  └─────────────┘    └──────────────┘    └─────────────────┘    │
│                              │                     │            │
│                              ▼                     ▼            │
│                       ┌─────────────────┐    ┌──────────────┐   │
│                       │  Query Vector   │    │ Top-k Most   │   │
│                       │  (Same Space)   │    │ Similar Docs │   │
│                       └─────────────────┘    └──────────────┘   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Similarity Metrics:
- **Cosine Similarity**: Measures angle between vectors (most common)
- **Euclidean Distance**: Measures straight-line distance
- **Dot Product**: Simple vector multiplication

### Popular Embedding Models:
| Model | Dimensions | Best For |
|-------|------------|----------|
| text-embedding-3-small | 1536 | General purpose, cost-effective |
| text-embedding-3-large | 3072 | High accuracy needs |
| sentence-transformers/all-MiniLM | 384 | On-device, fast |
| voyage-2 | 1024 | Enterprise RAG |

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai numpy scikit-learn

In [ ]:
import os
from getpass import getpass
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

# Setup API
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding vector for text"""
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return np.array(response.data[0].embedding)

def get_embeddings_batch(texts, model="text-embedding-3-small"):
    """Get embeddings for multiple texts"""
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [np.array(d.embedding) for d in response.data]

## 💡 Basic Example

Simple semantic search implementation.

In [ ]:
# Sample document collection
documents = [
    "The quick brown fox jumps over the lazy dog",
    "A fast auburn canine leaps across a sleepy hound",
    "Machine learning algorithms process data efficiently",
    "Neural networks learn patterns from large datasets",
    "The capital of France is Paris, known for the Eiffel Tower",
    "Paris, France's largest city, features iconic landmarks",
    "Python is a popular programming language for data science",
    "JavaScript powers interactive web applications",
    "Coffee contains caffeine which helps people stay awake",
    "Tea has less caffeine than coffee but provides calm alertness"
]

print("Building semantic index...")
document_embeddings = get_embeddings_batch(documents)
print(f"Indexed {len(documents)} documents\n")

def semantic_search(query, documents, doc_embeddings, top_k=3):
    """Perform semantic search"""
    # Get query embedding
    query_embedding = get_embedding(query)
    
    # Calculate similarities
    similarities = cosine_similarity(
        [query_embedding],
        doc_embeddings
    )[0]
    
    # Get top-k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'document': documents[idx],
            'score': float(similarities[idx]),
            'index': idx
        })
    
    return results

# Test queries
test_queries = [
    "fast animal jumping",
    "AI and deep learning",
    "famous French landmarks",
    "caffeinated beverages"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    results = semantic_search(query, documents, document_embeddings, top_k=3)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Score: {result['score']:.4f}")
        print(f"   Doc: '{result['document']}'")

## 🌍 Real-World Example

Customer support knowledge base with semantic search.

In [ ]:
# Customer support knowledge base
support_articles = [
    {
        "id": "KB001",
        "title": "Reset Your Password",
        "content": "To reset your password, click 'Forgot Password' on the login page. Enter your email address and check your inbox for a reset link. The link expires in 24 hours."
    },
    {
        "id": "KB002",
        "title": "Update Payment Method",
        "content": "Go to Account Settings > Billing > Payment Methods. Click 'Add New Card' and enter your details. You can set a new card as default for future payments."
    },
    {
        "id": "KB003",
        "title": "Cancel Subscription",
        "content": "Navigate to Account Settings > Subscription > Cancel. Your access continues until the end of your billing period. No refunds for partial months."
    },
    {
        "id": "KB004",
        "title": "Two-Factor Authentication Setup",
        "content": "Enable 2FA in Security Settings. Scan the QR code with an authenticator app like Google Authenticator or Authy. Save your backup codes securely."
    },
    {
        "id": "KB005",
        "title": "Export Your Data",
        "content": "Request a data export from Privacy Settings > Data Export. You'll receive a download link via email within 48 hours. Data is provided in JSON format."
    },
    {
        "id": "KB006",
        "title": "API Rate Limits",
        "content": "Free tier: 100 requests/hour. Pro tier: 10,000 requests/hour. Enterprise: Custom limits. Rate limit headers included in all API responses."
    },
    {
        "id": "KB007",
        "title": "Troubleshoot Login Issues",
        "content": "Clear browser cache and cookies. Disable browser extensions. Try incognito mode. Ensure your account isn't locked due to failed attempts."
    },
    {
        "id": "KB008",
        "title": "Change Email Address",
        "content": "Go to Account Settings > Profile > Email. Enter new email and confirm with password. Verification required before changes take effect."
    }
]

# Build search index
print("Building support knowledge base index...")
article_texts = [f"{a['title']}: {a['content']}" for a in support_articles]
article_embeddings = get_embeddings_batch(article_texts)
print(f"Indexed {len(support_articles)} articles\n")

def search_support(query, top_k=3):
    """Search support knowledge base"""
    results = semantic_search(query, article_texts, article_embeddings, top_k)
    
    # Enrich with article metadata
    for r in results:
        r['article'] = support_articles[r['index']]
    
    return results

# Test with natural language queries
support_queries = [
    "I forgot my login credentials",
    "How do I stop my monthly payments?",
    "My account got hacked, need extra security",
    "Can't sign in to my account",
    "Want to download all my information"
]

for query in support_queries:
    print(f"\n{'='*60}")
    print(f"Customer: '{query}'")
    print(f"{'='*60}")
    
    results = search_support(query, top_k=2)
    
    print("\nTop Articles Found:")
    for i, result in enumerate(results, 1):
        article = result['article']
        print(f"\n{i}. [{article['id']}] {article['title']} (score: {result['score']:.3f})")
        print(f"   {article['content'][:100]}...")

## ❌ Failure Case

When semantic search fails and limitations to be aware of.

In [ ]:
# Demonstrating semantic search limitations

# Create a problematic document set
problematic_docs = [
    "The bank of the river was steep and muddy",
    "I need to visit the bank to withdraw money",
    "The pilot sat on the bank of the aircraft",
    "Food bank donations help feed the hungry",
    "The blood bank needs more O-negative donors"
]

problematic_embeddings = get_embeddings_batch(problematic_docs)

print("=== LIMITATION 1: POLYSEMY (Multiple Meanings) ===")
query1 = "financial institution"
results1 = semantic_search(query1, problematic_docs, problematic_embeddings, top_k=3)
print(f"Query: '{query1}'\n")
for r in results1:
    print(f"Score: {r['score']:.4f} | '{r['document']}'")
print("\n⚠️ Note: 'river bank' may score similarly to 'financial bank'\n")

print("=== LIMITATION 2: DOMAIN-SPECIFIC TERMS ===")
technical_docs = [
    "Java is a programming language",
    "Java is an island in Indonesia",
    "Java coffee comes from Indonesian plantations"
]
technical_embeddings = get_embeddings_batch(technical_docs)
query2 = "object-oriented programming"
results2 = semantic_search(query2, technical_docs, technical_embeddings, top_k=3)
print(f"Query: '{query2}'\n")
for r in results2:
    print(f"Score: {r['score']:.4f} | '{r['document']}'")
print("\n⚠️ Note: Without 'programming' context, 'Java' ambiguity persists\n")

print("=== LIMITATION 3: NEGATION HANDLING ===")
negation_docs = [
    "The product is excellent and highly recommended",
    "The product is not excellent and not recommended"
]
negation_embeddings = get_embeddings_batch(negation_docs)
query3 = "good product recommendation"
results3 = semantic_search(query3, negation_docs, negation_embeddings, top_k=2)
print(f"Query: '{query3}'\n")
for r in results3:
    print(f"Score: {r['score']:.4f} | '{r['document']}'")
print("\n⚠️ Note: Negation may not be properly captured in embeddings\n")

print("=== SOLUTIONS ===")
print("""
1. Hybrid Search: Combine semantic + keyword search
2. Query Expansion: Add context to disambiguate
3. Metadata Filtering: Pre-filter by category/domain
4. Re-ranking: Use cross-encoders for final ranking
5. Fine-tuning: Train embeddings on domain data
""")

## 📊 Benchmark Comparison

| Search Method | Keyword Match | Semantic Match | Speed | Complexity |
|---------------|---------------|----------------|-------|------------|
| **Keyword (BM25)** | 95% | 20% | Very Fast | Low |
| **Semantic Only** | 40% | 90% | Medium | Medium |
| **Hybrid** | 90% | 88% | Medium | High |
| **Dense + Sparse** | 92% | 92% | Medium | High |

### Embedding Model Comparison:
| Model | MTEB Avg | Cost | Speed | Dimensions |
|-------|----------|------|-------|------------|
| text-embedding-3-small | 62.3% | $0.02/M | Fast | 1536 |
| text-embedding-3-large | 64.6% | $0.13/M | Medium | 3072 |
| voyage-2 | 69.0% | $0.10/M | Medium | 1024 |
| all-MiniLM-L6-v2 | 56.0% | Free | Fast | 384 |

### Key Insights:
- Semantic search excels at conceptual understanding
- Keyword search better for exact matches and IDs
- Hybrid approaches typically outperform either alone
- Embedding quality directly impacts search quality

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              SEMANTIC SEARCH EXPERIMENT LAB                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Semantic Search Playground\n")

# Pre-defined document sets
doc_sets = {
    "1": {"name": "Animals", "docs": [
        "Dogs are loyal companions and great family pets",
        "Cats are independent and clean animals",
        "Birds can be taught to speak and sing",
        "Fish are low-maintenance pets for small spaces",
        "Horses require large areas and daily exercise"
    ]},
    "2": {"name": "Technology", "docs": [
        "Cloud computing enables scalable infrastructure",
        "Artificial intelligence transforms industries",
        "Blockchain provides decentralized trust",
        "Cybersecurity protects digital assets",
        "IoT connects everyday devices to the internet"
    ]},
    "3": {"name": "Custom", "docs": []}
}

print("Choose document set:")
for key, value in doc_sets.items():
    if key != "3":
        print(f"{key}. {value['name']}")
print("3. Enter custom documents")

choice = input("\nEnter choice (1-3): ")

if choice == "3":
    print("\nEnter your documents (one per line, empty line to finish):")
    custom_docs = []
    while True:
        doc = input()
        if doc == "":
            break
        custom_docs.append(doc)
    selected_docs = custom_docs
else:
    selected_docs = doc_sets[choice]['docs']

print(f"\nIndexing {len(selected_docs)} documents...")
selected_embeddings = get_embeddings_batch(selected_docs)
print("Index built!\n")

# Search loop
while True:
    query = input("\nEnter search query (or 'quit' to exit): ")
    if query.lower() == 'quit':
        break
    
    top_k = int(input("Number of results (default 3): ") or "3")
    
    results = semantic_search(query, selected_docs, selected_embeddings, top_k)
    
    print(f"\n{'='*60}")
    print(f"Results for: '{query}'")
    print(f"{'='*60}")
    
    for i, r in enumerate(results, 1):
        print(f"\n{i}. Score: {r['score']:.4f}")
        print(f"   {r['document']}")

## 💡 Tips & Tricks

### Optimization Strategies:

**1. Query Enhancement:**
- Expand short queries with context
- Use HyDE (Hypothetical Document Embeddings)
- Add domain-specific keywords

**2. Index Optimization:**
- Use appropriate chunk sizes (200-500 tokens)
- Include metadata in embeddings (title + content)
- Normalize text before embedding

**3. Post-Processing:**
- Apply similarity thresholds
- Use MMR (Maximal Marginal Relevance) for diversity
- Re-rank with cross-encoders

### Common Pitfalls:
- ❌ Chunks too large → diluted embeddings
- ❌ Chunks too small → missing context
- ❌ No text normalization → inconsistent results
- ❌ Wrong embedding model for domain
- ❌ Ignoring similarity thresholds

### Vector Database Options:
| Database | Best For | Features |
|----------|----------|----------|
| **Pinecone** | Production | Managed, scalable |
| **Weaviate** | Hybrid search | Graph + vector |
| **Chroma** | Prototyping | Easy setup |
| **FAISS** | Research | Fast, local |
| **pgvector** | Existing PG | SQL integration |

## 📚 References

### Research:
- [Dense Passage Retrieval for Open-Domain QA (Karpukhin et al., 2020)](https://arxiv.org/abs/2004.04906)
- [Sentence-BERT: Sentence Embeddings (Reimers & Gurevych, 2019)](https://arxiv.org/abs/1908.10084)
- [MTEB: Massive Text Embedding Benchmark (Muennighoff et al., 2022)](https://arxiv.org/abs/2210.07316)

### Documentation:
- [OpenAI Embeddings API](https://platform.openai.com/docs/guides/embeddings)
- [Sentence Transformers](https://www.sbert.net/)
- [FAISS Documentation](https://github.com/facebookresearch/faiss/wiki)

### Related Techniques:
- Basic RAG (Technique 53)
- Document Chunking (Technique 55)
- Hybrid Retrieval (Technique 57)
- Re-Ranking (Technique 58)